<style>
@import url('https://fonts.googleapis.com/css2?family=Literata:opsz,wght@7..72,400;7..72,500;7..72,600;7..72,700&family=JetBrains+Mono:wght@400;500&display=swap');

.md-wrap, .jp-RenderedMarkdown, .rendered_html {
  font-family: 'Literata', Georgia, serif !important;
}
.md-wrap code, .jp-RenderedMarkdown code, .rendered_html code {
  font-family: 'JetBrains Mono', monospace !important;
}

.note, .finding, .warn, .fail {
  border-radius: 6px;
  padding: 14px 18px;
  margin: 14px 0;
  font-family: 'Literata', Georgia, serif;
  line-height: 1.55;
  border-left: 4px solid;
}
.note   { background: #EAF3F6; border-color: #2A6274; }
.finding{ background: #EEF3E8; border-color: #4B6B2F; }
.warn   { background: #FBF1DE; border-color: #B0791A; }
.fail   { background: #FBEAE6; border-color: #A6402F; }
.note b, .finding b, .warn b, .fail b { font-family: 'JetBrains Mono', monospace; font-weight: 600; }
</style>

# Soil Condition Along the Kazakhstan Transect: A Predictive Modeling Study

This notebook analyzes a soil dataset collected along a roughly 1,300&nbsp;km
transect in Kazakhstan, extending from Petropavlovsk in the north to Taraz in
the south. The transect spans three climatic zones and includes two sampling
campaigns (May and September 2015) across 40 sites. A full laboratory panel
is available for every site: organic and inorganic carbon, total nitrogen,
pH, electrical conductivity, bulk density, and soil moisture.

**Working hypothesis.** Precipitation and temperature along the transect
should, by themselves, carry substantial predictive information about soil
condition — that is, a simple model given only climate and terrain features
should already perform reasonably well. If this holds, it becomes worth
testing whether a genetic algorithm can automatically identify a compact,
transferable feature subset, comparable to or better than manual feature
selection.

<div class="note">
<b>Why two validation schemes?</b> Because sites lie along a single line and
climate varies smoothly along it, a naive Leave-One-Site-Out (LOSO) scheme is
vulnerable to leakage through spatial autocorrelation: the left and right
neighbors of a held-out site are almost certainly similar to it, so a model
can appear to generalize by exploiting proximity rather than by learning the
underlying relationship. To guard against this, every model reported here is
also evaluated on held-out <i>latitudinal blocks</i> — contiguous stretches
of the transect never seen during training. This second score is typically
far less flattering, and is treated as the more informative of the two.
</div>

**Requirements to run.** Two files are attached as a Kaggle Dataset:
`Supplement 2.xlsx` (the raw supplement) and `site_climate.csv` — the reason
a separate climate file was necessary is explained in the next section.

**Runtime.** A full run takes 10-20 minutes on a standard 4-core Kaggle CPU.
Accuracy was not traded for speed: where the genetic algorithm requires an
exhaustive search over candidate configurations, it performs one.


## Imports and data location

The dependencies are standard: scikit-learn, XGBoost, and matplotlib. The one
implementation detail worth noting is that the input path is not hardcoded,
since on Kaggle it depends on how the dataset is named at upload time;
resolving it dynamically lets the notebook run unmodified for anyone who
forks it.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from sklearn.ensemble import (GradientBoostingRegressor, RandomForestClassifier,
                              RandomForestRegressor, StackingRegressor)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             cohen_kappa_score, confusion_matrix, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.model_selection import (GroupKFold, LeaveOneGroupOut,
                                     cross_val_predict)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from xgboost import XGBClassifier, XGBRegressor

# LOSO produces 40 folds, and scikit-learn tends to emit convergence and
# division warnings on degenerate folds; these are suppressed to keep the
# output readable
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#CFD1C4", "axes.grid": True,
    "grid.color": "#E4E5DA", "grid.linewidth": 0.8,
    "font.size": 11, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
})
ACCENT, ACCENT_DIM, WARN = "#245A6B", "#6E97A3", "#8A5215"
STATE_COLORS = {"critical": "#E8D5A9", "poor": "#C99A52",
                "moderate": "#8C6234", "healthy": "#3A2E1E"}


In [ ]:
def find_input(name: str) -> Path:
    # local run: the file sits next to the notebook; on Kaggle it appears
    # under /kaggle/input/<dataset-slug>/, and the slug is not known in
    # advance, so every subfolder is searched
    local = Path(name)
    if local.exists():
        return local
    kin = Path("/kaggle/input")
    if kin.exists():
        for sub in sorted(kin.iterdir()):
            cand = sub / name
            if cand.exists():
                return cand
    raise FileNotFoundError(
        f"could not find '{name}' — on Kaggle attach the dataset via "
        f"Add Input; for a local run, place the file next to the notebook")


SRC = find_input("Supplement 2.xlsx")
CLIMATE_PATH = find_input("site_climate.csv")
print(f"found: {SRC}")
print(f"found: {CLIMATE_PATH}")


## First, what is actually inside this spreadsheet

The file was inspected manually before writing any parsing code, and it
quickly became clear that this is not a pre-cleaned Kaggle CSV. Nine sheets
are present, each covering a different measurement type; headers span two to
three rows, and units appear inconsistently — sometimes inside the column
name, sometimes on a separate row beneath it. Each sheet is therefore parsed
individually, which is more reliable than attempting a single universal
parser.

<div class="warn">
One issue surfaced immediately: the text value <code>"missing"</code>
appears in a numeric EC column (site 8, September measurement).
<code>pd.to_numeric(..., errors="coerce")</code> converts it to NaN instead
of failing the parse for the entire sheet.
</div>


In [ ]:
def _sid(s: pd.Series) -> pd.Series:
    # site id is inconsistently typed across sheets (float in some, object
    # in others); casting to a single dtype up front prevents the merges
    # below from silently dropping rows
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def load_annotation() -> pd.DataFrame:
    ann = pd.read_excel(SRC, sheet_name="Annotation (9)")
    ann = ann.rename(columns={
        "Internal sample #": "sample_id", "Location": "location",
        "Site Description": "site_description", "Land Use /Cover": "land_use",
        "Elevation, meters (msl)": "elevation_m", "Biome ": "biome",
        "Environmental Feature": "env_feature", "Soil type": "soil_type_wrb",
        "Longtitude": "longitude", "Latitude ": "latitude",
        "Mean total precipitation, mm": "precip_mm",
        "Mean annual temperature, ºC": "mat_c",
    })
    ann["sample_id"] = _sid(ann["sample_id"])
    # categories were entered manually in the spreadsheet — 'chernozem ' and
    # 'chernozem' are distinct values to pandas but not to an agronomist
    for c in ["land_use", "biome", "env_feature", "soil_type_wrb", "location"]:
        ann[c] = ann[c].astype(str).str.strip().str.lower()
    ann["biome"] = ann["biome"].str.replace(r"\s+", " ", regex=True)
    ann["env_feature"] = ann["env_feature"].str.replace(r"\s+", " ", regex=True)
    ann["soil_type_wrb"] = ann["soil_type_wrb"].str.replace(r"\s*\+\s*", "+", regex=True)
    return ann[["sample_id", "location", "land_use", "elevation_m", "biome",
                "env_feature", "soil_type_wrb", "longitude", "latitude",
                "precip_mm", "mat_c"]]


def load_carbon() -> pd.DataFrame:
    # the header spans three rows (units / method / STDEV-SE); extracting
    # columns by position is more robust than parsing this header structure
    raw = pd.read_excel(SRC, sheet_name="TC,TOC,TIC (1) ", header=None, skiprows=3)
    cols = {0: "sample_id", 1: "tc_may", 4: "toc_may", 7: "tic_may",
            10: "tc_sep", 13: "toc_sep", 16: "tic_sep"}
    df = raw[list(cols)].rename(columns=cols)
    df["sample_id"] = _sid(df["sample_id"])
    return df.dropna(subset=["sample_id"])


def load_loi() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="LOI (2)").rename(columns={
        "Sample #": "sample_id", "LOI, %(MAY)": "loi_may", "LOI, % (SEP)": "loi_sep"})
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "loi_may", "loi_sep"]]


def load_nitrogen() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="Total Nitrogen (3)", skiprows=[1]).rename(columns={
        "TN": "sample_id", "Mean May": "tn_may", "Mean Sep": "tn_sep"})
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "tn_may", "tn_sep"]]


def load_bulk_density() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="Dry bulk Density (4)").rename(columns={
        "Sample #": "sample_id",
        "Dry bulk density, g/kg (MAY)": "bd_may",
        "Dry bulk density, g/kg, LOI(SEP)": "bd_sep"})
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "bd_may", "bd_sep"]]


def load_moisture() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="Soil Moisture  (5)")
    df.columns = ["sample_id", "dt_may", "sm_grav_may", "_s1", "sm_vol_may",
                  "dt_sep", "sm_grav_sep", "_s2", "sm_vol_sep"]
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "dt_may", "sm_grav_may", "sm_vol_may",
               "dt_sep", "sm_grav_sep", "sm_vol_sep"]]


def load_ph() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="pH SU & SN (6)").rename(columns={
        "Sample #": "sample_id",
        "pH SN   (MAY)": "ph_sn_may", "pH SN   (SEP)": "ph_sn_sep",
        "pH SU   (MAY)": "ph_su_may", "pH SU   (SEP)": "ph_su_sep"})
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "ph_sn_may", "ph_sn_sep", "ph_su_may", "ph_su_sep"]]


def load_ec() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="EC SN (7)").rename(columns={
        "Sample #": "sample_id",
        "EC SN   (MAY), μS/cm": "ec_may", "EC SN   (SEP), μS/cm": "ec_sep"})
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "ec_may", "ec_sep"]]


def load_stocks() -> pd.DataFrame:
    df = pd.read_excel(SRC, sheet_name="TOC and TN stocks (8)", skiprows=[1]).rename(columns={
        "Samples": "sample_id", "Total organic carbon": "toc_stock_t_ha",
        "Total nitrogen": "tn_stock_t_ha"})
    df["sample_id"] = _sid(df["sample_id"])
    return df[["sample_id", "toc_stock_t_ha", "tn_stock_t_ha"]]


SEASONAL = ["tc", "toc", "tic", "loi", "tn", "bd", "sm_grav", "sm_vol",
            "ph_sn", "ph_su", "ec"]

print("all nine sheets loaded")


### Climate data required a separate source

The annotation sheet was loaded separately, and an irregularity emerged: for
neighboring sites, precipitation and temperature values sometimes match to
every decimal place. This is not coincidence — the values were assigned to
blocks of sites rather than to individual sites, apparently sourced from a
coarse-resolution climate atlas.

<div class="fail">
Training on these block-level values and then, in production, supplying
climate data from a proper service such as Open-Meteo would create a
systematic mismatch between training and inference inputs. Cross-validation
metrics would remain favorable, since the dataset is internally consistent,
but this is precisely the situation in which strong CV scores fail to
reflect real-world performance.
</div>

To avoid this, the 1991-2020 WMO climate normals were retrieved separately
from the ERA5 archive (via Open-Meteo) for each of the 40 sites, using their
actual coordinates rather than block-level rounding. This is the second
input file, `site_climate.csv`.


In [ ]:
climate = pd.read_csv(CLIMATE_PATH)
# a remaining fallback value would indicate that the ERA5 request failed for
# some sites; in that case it is preferable to fail explicitly rather than
# silently train on placeholder values
assert not climate["climate_is_fallback"].any(), (
    "site_climate.csv contains sites without ERA5 climate data — re-fetch the file")
print(f"climate data available for all {len(climate)} sites, fallbacks: "
      f"{int(climate['climate_is_fallback'].sum())}")
climate.head()


In [ ]:
def build_dataset() -> pd.DataFrame:
    wide = load_annotation()
    for loader in (load_carbon, load_loi, load_nitrogen, load_bulk_density,
                   load_moisture, load_ph, load_ec, load_stocks):
        wide = wide.merge(loader(), on="sample_id", how="left")

    # each sheet stores May and September as separate columns; reshaping to
    # long format so that a row represents "site + season" rather than "site"
    frames = []
    for season, suffix in (("may", "may"), ("sep", "sep")):
        part = wide[[c for c in wide.columns
                     if not c.endswith(("_may", "_sep"))]].copy()
        part["season"] = season
        for base in SEASONAL:
            part[base] = pd.to_numeric(wide[f"{base}_{suffix}"], errors="coerce")
        part["sampling_dt"] = pd.to_datetime(wide[f"dt_{suffix}"], errors="coerce")
        frames.append(part)

    df = pd.concat(frames, ignore_index=True)

    # block-level climate from the original supplement is renamed to _paper
    # and retained only for reference; training uses the ERA5-derived values
    df = df.merge(climate, on="sample_id", how="left")
    df = df.rename(columns={"precip_mm": "precip_mm_paper",
                            "mat_c": "mat_c_paper",
                            "elevation_m": "elevation_m_paper"})
    df["precip_mm"] = df["era5_precip_mm"]
    df["mat_c"] = df["era5_mat_c"]
    df["elevation_m"] = df["era5_elevation_m"]

    df["doy"] = df["sampling_dt"].dt.dayofyear
    df["is_september"] = (df["season"] == "sep").astype(int)
    # C:N ratio: a standard indicator of organic matter mineralization rate,
    # retained as a potential feature
    df["cn_ratio"] = df["toc"] / df["tn"].replace(0, np.nan)
    # carbonate fraction of total carbon: an indirect indicator of saline
    # or alkaline soils
    df["carbonate_frac"] = df["tic"] / df["tc"].replace(0, np.nan)

    return df.sort_values(["sample_id", "season"]).reset_index(drop=True)


raw = build_dataset()
print(f"resulting dataset: {raw.shape[0]} rows x {raw.shape[1]} columns "
      f"({raw['sample_id'].nunique()} sites, each sampled twice)")
raw[["toc", "tn", "sm_grav", "ph_sn", "ec", "bd", "precip_mm", "mat_c"]].describe().round(2).T


## Feature preparation

Nine WRB soil subtypes across forty sites is excessive for a sample of this
size; several classes would contain observations from only one or two sites.
Subtypes are therefore consolidated into five agronomically similar groups.
Electrical conductivity is also log-transformed: it ranges from 60 to
9176&nbsp;μS/cm, and without the transform a single saline outlier would
dominate the scale for linear models.


In [ ]:
SOIL_GROUP_MAP = {
    "chernozem": "chernozem", "chernozem+solonetz": "chernozem",
    "kastanozem": "kastanozem", "arenosol": "arenosol",
    "arenosol+solonetz": "arenosol", "calcisol+solonetz": "saline",
    "solonchak+solonetz": "saline", "regosol": "other", "umbrisol": "other",
}


def load_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["soil_group"] = df["soil_type_wrb"].map(SOIL_GROUP_MAP).fillna("other")
    df["has_solonetz"] = df["soil_type_wrb"].str.contains("solonetz").astype(int)
    df["ec_log"] = np.log10(df["ec"])
    df["ec_ds_m"] = df["ec"] / 1000.0  # μS/cm -> dS/m, a more familiar agronomic scale
    # De Martonne aridity index: a single value in place of two correlated
    # features (precipitation and temperature move together, so combining
    # them avoids a second, redundant predictor)
    df["aridity"] = df["precip_mm"] / (df["mat_c"] + 10.0)
    # the single missing value (the "missing" placeholder in EC) is imputed
    # with the median rather than dropping the site entirely; losing one of
    # 40 sites is costly
    for col in ["ec", "ec_log", "ec_ds_m"]:
        df[col] = df[col].fillna(df[col].median())
    return df


df = load_dataset(raw)


## Target variable construction

No target column exists in the raw dataset — only raw measurements — yet the
target application requires a four-class soil condition contract
(`critical / poor / moderate / healthy`). This label therefore has to be
constructed from the available measurements.

An agronomic score (0-18) was assembled in which carbon and nitrogen are
weighted twice as heavily as pH and salinity, since organic matter is the
primary determinant of fertility and cannot be measured with a low-cost
field sensor, unlike pH and EC.

<div class="warn">
This construction has an important consequence: because pH and EC enter the
scoring formula directly, any model given these two features as predictors
will recover part of the label for free. This is verified explicitly further
below, where a model trained on pH and EC alone is used to quantify how much
of the apparent classification performance is attributable to this
formula-level leakage rather than genuine signal.
</div>


In [ ]:
SOIL_STATE_CLASSES = ["critical", "poor", "moderate", "healthy"]
SOIL_STATE_CUTS = (6, 10, 14)  # score boundaries (0..18) between classes,
                               # chosen to keep class sizes reasonably balanced


def _score_row(r) -> int:
    toc = 0 if r.toc < 6 else 1 if r.toc < 12 else 2 if r.toc < 20 else 3
    tn = 0 if r.tn < 0.75 else 1 if r.tn < 1.25 else 2 if r.tn < 2.0 else 3
    p = r.ph_sn
    if 6.0 <= p <= 7.5:
        ph = 3
    elif 5.5 <= p < 6.0 or 7.5 < p <= 8.0:
        ph = 2
    elif 5.0 <= p < 5.5 or 8.0 < p <= 8.5:
        ph = 1
    else:
        ph = 0
    e = r.ec_ds_m
    ec = 3 if e < 0.5 else 2 if e < 1.0 else 1 if e < 2.0 else 0
    return 2 * toc + 2 * tn + ph + ec


df["soil_score"] = df.apply(_score_row, axis=1)
df["soil_state"] = pd.cut(df["soil_score"], [-1, *SOIL_STATE_CUTS, 999],
                          labels=SOIL_STATE_CLASSES).astype(str)
df["soil_state_idx"] = df["soil_state"].map(
    {c: i for i, c in enumerate(SOIL_STATE_CLASSES)})

df["soil_state"].value_counts().reindex(SOIL_STATE_CLASSES)


### Visual inspection of the full transect

Before fitting any model, the 40 sites are examined visually. Sites are
ordered by latitude, and climate variables are overlaid with the resulting
soil-condition class for both seasons.


In [ ]:
sites = (df.drop_duplicates("sample_id")
         .sort_values("latitude", ascending=False)
         .reset_index(drop=True))
piv_state = df.pivot_table(index="sample_id", columns="season",
                           values="soil_state", aggfunc="first")
sites = sites.merge(piv_state.rename(columns={"may": "state_may", "sep": "state_sep"}),
                    on="sample_id")

fig, axes = plt.subplots(4, 1, figsize=(13, 8.5), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1.3, 0.55, 0.55],
                                      "hspace": 0.12})

ax = axes[0]
ax.fill_between(range(len(sites)), sites["precip_mm"], color=ACCENT, alpha=0.15)
ax.plot(range(len(sites)), sites["precip_mm"], color=ACCENT, lw=2)
ax.set_ylabel("Precipitation, mm/yr")
ax.set_title("Climate and soil condition along the transect: Petropavlovsk → Taraz",
            loc="left")
imin, imax = sites["precip_mm"].idxmin(), sites["precip_mm"].idxmax()
for i, lab in [(imin, "minimum"), (imax, "maximum")]:
    ax.annotate(f"{sites['precip_mm'][i]:.0f} mm · {lab}",
               (i, sites["precip_mm"][i]), textcoords="offset points",
               xytext=(0, 8), ha="center", fontsize=9, color="#3F4238")

ax = axes[1]
ax.plot(range(len(sites)), sites["mat_c"], color="#8C6234", lw=2)
ax.set_ylabel("Mean annual t, °C")

for ax, col, label in [(axes[2], "state_may", "May"), (axes[3], "state_sep", "Sep")]:
    colors = [STATE_COLORS[s] for s in sites[col]]
    ax.bar(range(len(sites)), [1] * len(sites), color=colors, width=1.0,
          edgecolor="white", linewidth=0.6)
    ax.set_yticks([])
    ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=10)
    ax.grid(False)

axes[-1].set_xticks(range(0, len(sites), 5))
axes[-1].set_xticklabels([f"{sites['latitude'][i]:.1f}°" for i in range(0, len(sites), 5)])
axes[-1].set_xlabel("Latitude, °N (north → south)")

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in STATE_COLORS.values()]
fig.legend(handles, STATE_COLORS.keys(), loc="upper center", ncol=4,
          bbox_to_anchor=(0.5, 1.015), frameon=False, fontsize=10)
plt.tight_layout()
plt.show()


<div class="finding">
The precipitation trough below 150&nbsp;mm around Lake Balkhash (roughly the
midpoint of the transect) coincides closely with a continuous band of
<code>critical</code> classifications in both bottom rows. With only forty
sites, a formal causal claim is not warranted, but the visual pattern is
clear enough to expect that a model would recover much of its predictive
signal from this relationship — and, as shown below, this is indeed the
case: precipitation is among the features selected for the nitrogen model.
</div>

A further observation: the soil-condition class changes between May and
September for 14 of the 40 sites. The dataset therefore captures genuine
seasonal dynamics rather than a single static snapshot, which a model could
potentially learn rather than merely memorize per site.


## Excluding features that would leak the target

Several columns in this dataset are not independent measurements but
arithmetic derivatives of one another. Left unaddressed, a model would
effectively predict the target from a transformation of itself, producing
excellent metrics that carry no real information.

- volumetric moisture = gravimetric moisture × bulk density — the same
  measurement in different units;
- total carbon = organic + inorganic carbon;
- loss on ignition is, in essence, also a measure of organic matter,
  obtained by a different method;
- stocks (t/ha) = concentration × bulk density × depth;
- C:N is a ratio computed directly from the target variable.

Going forward, only quantities that a field sensor could plausibly provide
at prediction time are retained: pH, EC, moisture, soil temperature, plus
coordinates and climate normals.


In [ ]:
LEAKAGE = {
    "sm_grav": ["sm_vol"],
    "tn": ["tn_stock_t_ha", "cn_ratio", "toc", "tc", "tic", "loi",
           "toc_stock_t_ha", "carbonate_frac"],
    "toc": ["toc_stock_t_ha", "cn_ratio", "tn", "tc", "tic", "loi",
            "tn_stock_t_ha", "carbonate_frac"],
    # ph_su is the same acidity measurement obtained via a different
    # extraction method; supplying it to the model would amount to
    # predicting pH from pH
    "ph_sn": ["ph_su", "soil_score", "soil_state"],
    "soil_state_idx": ["toc", "tn", "tc", "tic", "loi", "cn_ratio",
                       "toc_stock_t_ha", "tn_stock_t_ha", "carbonate_frac",
                       "soil_score", "soil_state", "sm_vol"],
}

SENSOR_FEATURES = ["ph_sn", "ec_log", "sm_grav", "bd"]
CLIMATE_FEATURES = ["precip_mm", "mat_c", "aridity"]
TERRAIN_FEATURES = ["elevation_m", "latitude", "longitude"]
SEASON_FEATURES = ["is_september", "doy"]
CATEGORICAL_FEATURES = ["land_use", "soil_group"]
BINARY_FEATURES = ["has_solonetz"]


def encode(df: pd.DataFrame, numeric: list, categorical: list = None) -> pd.DataFrame:
    parts = [df[numeric].astype(float)]
    if categorical:
        for col in categorical:
            parts.append(pd.get_dummies(df[col], prefix=col).astype(float))
    out = pd.concat(parts, axis=1)
    # column order is fixed once and for all; otherwise one-hot encoding from
    # separate calls could produce columns in a different order, and the
    # model would silently misassign features
    return out.reindex(sorted(out.columns), axis=1)


## Validation scheme, and why plain K-fold is unsuitable

Standard k-fold cross-validation fails here immediately: the May and
September samples from the same site are, for practical purposes, the same
soil measured twice. A plain `KFold` split would readily place both halves
of a site in train and test simultaneously, and the model would appear to
"predict" observations it has effectively already seen. The primary scheme
is therefore **Leave-One-Site-Out (LOSO)**: forty folds, with an entire site
held out for testing in each.

As noted above, LOSO alone is insufficient given the spatial autocorrelation
along the transect. A second validation layer — **five latitudinal block
folds** — is therefore used: a contiguous segment of the transect is
withheld entirely, testing whether the model generalizes to a region it has
never observed even indirectly.


In [ ]:
def loso_cv():
    return LeaveOneGroupOut()


def spatial_blocks(df: pd.DataFrame, n_blocks: int = 5) -> np.ndarray:
    return pd.qcut(df["latitude"], n_blocks, labels=False).to_numpy()


def regression_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {"R2": float(r2_score(y_true, y_pred)), "RMSE": rmse,
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "RMSE/SD": float(rmse / y_true.std(ddof=1))}


def classification_metrics(y_true, y_pred) -> dict:
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_F1": float(f1_score(y_true, y_pred, average="macro")),
            "cohen_kappa": float(cohen_kappa_score(y_true, y_pred))}


def fmt_metrics(m: dict) -> str:
    return "  ".join(f"{k}={v:.3f}" for k, v in m.items())


def evaluate(model, X, y, groups, blocks, task="reg") -> dict:
    metric_fn = regression_metrics if task == "reg" else classification_metrics
    out = {}
    pred_site = cross_val_predict(model, X, y, groups=groups, cv=loso_cv(), n_jobs=-1)
    out["by_site"] = metric_fn(y, pred_site)
    pred_block = cross_val_predict(model, X, y, groups=blocks,
                                   cv=GroupKFold(n_splits=len(np.unique(blocks))), n_jobs=-1)
    out["by_region"] = metric_fn(y, pred_block)
    out["_pred_site"] = pred_site
    return out


groups = df["sample_id"].to_numpy()
blocks = spatial_blocks(df)
print(f"{len(df)} observations, {df['sample_id'].nunique()} sites (LOSO), "
      f"{len(np.unique(blocks))} latitudinal blocks")


## First target: soil moisture — starting with a simple baseline

Rather than reaching for XGBoost immediately, a plain linear regression on
climate, terrain, and a small set of sensor features is fit first. If this
already performs reasonably, the question of whether a more complex model is
warranted becomes worth asking explicitly.


In [ ]:
num_moist = (CLIMATE_FEATURES + TERRAIN_FEATURES + SEASON_FEATURES + BINARY_FEATURES
            + ["bd", "ph_sn", "ec_log"])
Xa = encode(df, num_moist, CATEGORICAL_FEATURES)
ya = df["sm_grav"].to_numpy()

baseline_moist = Pipeline([("scaler", StandardScaler()),
                           ("model", LinearRegression())])
res_baseline_moist = evaluate(baseline_moist, Xa, ya, groups, blocks, "reg")
print("linear regression, soil moisture:")
print(f"  by site:   {fmt_metrics(res_baseline_moist['by_site'])}")
print(f"  by region: {fmt_metrics(res_baseline_moist['by_region'])}")


<div class="fail">
An early warning sign: <b>R² = 0.44 by site, but -0.59 by region</b>. Within
the familiar transect the model captures some signal, but withholding an
entire latitudinal block makes it perform worse than simply predicting the
sample mean. The linear model is evidently extrapolating beyond its training
range, and doing so poorly. It is retained in the results table below as a
"worst case" reference, but a more flexible model is clearly needed.
</div>

A broader set of algorithms is compared next: linear models, decision trees,
random forest, KNN, and XGBoost. This set follows
[a repository addressing a similar soil-moisture prediction task](https://github.com/lokesh28-krish/Soil-Moisture-Prediction-Using-machine-Learning-algorithms)
— there is little value in re-deriving a reasonable starting algorithm set
from scratch.


In [ ]:
def _scaled(estimator):
    # linear models and KNN require feature scaling; otherwise EC (~10^3)
    # would dominate pH (~10^0) purely due to magnitude rather than actual
    # importance
    return Pipeline([("scaler", StandardScaler()), ("model", estimator)])


def regressors() -> dict:
    return {
        "LinearRegression": _scaled(LinearRegression()),
        "Ridge": _scaled(Ridge(alpha=10.0, random_state=RANDOM_STATE)),
        "DecisionTree": DecisionTreeRegressor(
            max_depth=4, min_samples_leaf=5, random_state=RANDOM_STATE),
        "RandomForest": RandomForestRegressor(
            n_estimators=400, max_depth=6, min_samples_leaf=3,
            max_features="sqrt", random_state=RANDOM_STATE, n_jobs=1),
        "KNN": _scaled(KNeighborsRegressor(n_neighbors=5, weights="distance")),
        "XGBoost": XGBRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, reg_alpha=0.5,
            min_child_weight=3, random_state=RANDOM_STATE, n_jobs=1, verbosity=0),
    }
    # tree depth and min_samples_leaf are kept deliberately small; with 80
    # rows, scikit-learn's defaults would memorize individual observations


def compare_zoo(name, X, y, groups, blocks, zoo, task="reg"):
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    key = "R2" if task == "reg" else "macro_F1"
    rows, preds = {}, {}
    for algo, model in zoo.items():
        res = evaluate(model, X, y, groups, blocks, task)
        preds[algo] = res.pop("_pred_site")
        rows[algo] = res
        print(f"  {algo:<18} by site:   {fmt_metrics(res['by_site'])}")
        print(f"  {'':<18} by region: {fmt_metrics(res['by_region'])}")
    best = max(rows, key=lambda a: rows[a]["by_site"][key])
    print(f"\n  -> selecting {best} ({key}={rows[best]['by_site'][key]:.3f})")
    return best, {"algorithms": rows, "best": best, "predictions": preds}


best_a, res_a = compare_zoo("SOIL MOISTURE, full model comparison", Xa, ya, groups, blocks,
                            regressors(), "reg")


<div class="finding">
RandomForest achieves R²=0.63 by site and 0.46 by region — not outstanding,
but a clear improvement over the linear model's -0.59 on the same split. The
gap between validation schemes remains substantial, so this is not yet a
settled result, but the model at least outperforms a constant baseline.
</div>


## Second target: total nitrogen — introducing the genetic algorithm

Before invoking the GA, the same baseline ritual applies: a linear
regression to calibrate expectations.


In [ ]:
num_nutrient = SENSOR_FEATURES + CLIMATE_FEATURES + TERRAIN_FEATURES + SEASON_FEATURES + BINARY_FEATURES
Xb = encode(df, num_nutrient, CATEGORICAL_FEATURES)
yb = df["tn"].to_numpy()

res_baseline_n = evaluate(Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
                          Xb, yb, groups, blocks, "reg")
print(f"linear regression, nitrogen: by site   {fmt_metrics(res_baseline_n['by_site'])}")
print(f"                             by region {fmt_metrics(res_baseline_n['by_region'])}")


<div class="fail">
R²=0.48 by site is passable at first glance. By region, however, it is
<b>-1.55</b> — worse than the moisture result above: the model is not merely
unhelpful, it performs roughly twice as poorly as the naive mean. The likely
cause is the same as before: the climatic gradient along the transect is
non-linear (recall the earlier figure — precipitation first declines, then
rises again toward the south), while a linear model must continue
extrapolating a straight line even where the underlying relationship
reverses direction.
</div>

The same algorithm set used for moisture is evaluated here as well.


In [ ]:
best_b, res_b = compare_zoo("TOTAL NITROGEN, full model comparison", Xb, yb, groups, blocks,
                            regressors(), "reg")


An improvement, but an open question remains: the model was given 23
features, and not all of them plausibly contribute — some are likely noise
that the forest memorizes as coincidental patterns in 80 rows. This
motivates testing a claim from the literature on genetic algorithms for NPK
prediction: whether a compact feature subset can be found **automatically**
that performs comparably, and transfers better to a new region.

<div class="note">
Implementation notes: this is not a textbook GA. It uses <b>rank-based
selection</b> instead of roulette-wheel selection (on a small sample, a
single high-fitness individual would otherwise dominate the population
immediately), <b>uniform crossover</b> instead of single-point, a
<b>decaying mutation rate</b> — broad exploration early, fine-tuning later —
and a <b>parsimony penalty</b> on feature count in the fitness function,
since a longer feature list is treated as increasingly likely to include
uninformative predictors.
</div>


In [ ]:
from dataclasses import dataclass, field
from typing import Callable


@dataclass
class Individual:
    mask: np.ndarray
    params: dict
    fitness: float = -np.inf
    raw_score: float = -np.inf


@dataclass
class EGAConfig:
    population_size: int = 30
    generations: int = 15
    elite_size: int = 3
    crossover_rate: float = 0.8
    mutation_rate_start: float = 0.25
    mutation_rate_end: float = 0.05
    parsimony: float = 0.004  # larger values impose a stronger penalty on feature-set size
    tournament_pressure: float = 1.8
    random_state: int = 42
    verbose: bool = True
    history: list = field(default_factory=list)


class EnhancedGA:
    def __init__(self, n_features, param_space, score_fn, config=None):
        self.n_features = n_features
        self.param_space = param_space
        self.param_names = list(param_space)
        self.score_fn = score_fn
        self.cfg = config or EGAConfig()
        self.rng = np.random.default_rng(self.cfg.random_state)
        self._cache = {}

    def _random_individual(self) -> Individual:
        # initial feature density is drawn randomly from [0.3, 0.8] so the
        # population immediately covers both sparse and dense feature sets
        density = self.rng.uniform(0.3, 0.8)
        mask = (self.rng.random(self.n_features) < density).astype(int)
        if mask.sum() == 0:
            mask[self.rng.integers(self.n_features)] = 1
        params = {k: self.rng.choice(v) for k, v in self.param_space.items()}
        return Individual(mask=mask, params=params)

    def _evaluate(self, ind: Individual) -> None:
        key = (tuple(ind.mask), tuple(sorted((k, str(v)) for k, v in ind.params.items())))
        if key in self._cache:  # the same mask/parameter combination can reappear after mutation or crossover
            ind.raw_score, ind.fitness = self._cache[key]
            return
        raw = self.score_fn(ind.mask.astype(bool), ind.params)
        penalty = self.cfg.parsimony * ind.mask.sum()
        fit = raw - penalty
        ind.raw_score, ind.fitness = raw, fit
        self._cache[key] = (raw, fit)

    def _rank_selection(self, pop, k):
        # rank rather than raw fitness: more robust to R2 occasionally
        # dropping sharply on a single unlucky fold with a small sample
        order = np.argsort([ind.fitness for ind in pop])
        n = len(pop)
        ranks = np.empty(n)
        ranks[order] = np.arange(1, n + 1)
        sp = self.cfg.tournament_pressure
        weights = (2 - sp) + 2 * (sp - 1) * (ranks - 1) / max(n - 1, 1)
        probs = weights / weights.sum()
        idx = self.rng.choice(n, size=k, p=probs)
        return [pop[i] for i in idx]

    def _crossover(self, a, b):
        if self.rng.random() > self.cfg.crossover_rate:
            return (Individual(a.mask.copy(), dict(a.params)),
                    Individual(b.mask.copy(), dict(b.params)))
        swap = self.rng.random(self.n_features) < 0.5
        m1, m2 = a.mask.copy(), b.mask.copy()
        m1[swap], m2[swap] = b.mask[swap], a.mask[swap]
        p1, p2 = dict(a.params), dict(b.params)
        for name in self.param_names:
            if self.rng.random() < 0.5:
                p1[name], p2[name] = b.params[name], a.params[name]
        return (Individual(self._repair(m1), p1), Individual(self._repair(m2), p2))

    def _mutate(self, ind, generation):
        # rate decays linearly from start to end: strong exploration early,
        # fine-tuning near the end
        t = generation / max(self.cfg.generations - 1, 1)
        rate = self.cfg.mutation_rate_start * (1 - t) + self.cfg.mutation_rate_end * t
        flips = self.rng.random(self.n_features) < rate
        ind.mask[flips] = 1 - ind.mask[flips]
        ind.mask = self._repair(ind.mask)
        for name, values in self.param_space.items():
            if self.rng.random() < rate:
                ind.params[name] = self.rng.choice(values)

    def _repair(self, mask):
        # an empty mask corresponds to an individual with nothing to train
        # on, and is repaired immediately
        if mask.sum() == 0:
            mask[self.rng.integers(self.n_features)] = 1
        return mask

    def run(self) -> Individual:
        pop = [self._random_individual() for _ in range(self.cfg.population_size)]
        for ind in pop:
            self._evaluate(ind)
        best = max(pop, key=lambda i: i.fitness)
        for gen in range(self.cfg.generations):
            pop.sort(key=lambda i: i.fitness, reverse=True)
            next_pop = [Individual(i.mask.copy(), dict(i.params), i.fitness, i.raw_score)
                       for i in pop[:self.cfg.elite_size]]  # the elite survives the generation unchanged
            parents = self._rank_selection(pop, self.cfg.population_size)
            i = 0
            while len(next_pop) < self.cfg.population_size:
                c1, c2 = self._crossover(parents[i % len(parents)], parents[(i + 1) % len(parents)])
                self._mutate(c1, gen)
                self._mutate(c2, gen)
                next_pop.extend([c1, c2])
                i += 2
            next_pop = next_pop[:self.cfg.population_size]
            for ind in next_pop:
                self._evaluate(ind)
            pop = next_pop
            gen_best = max(pop, key=lambda i: i.fitness)
            if gen_best.fitness > best.fitness:
                best = Individual(gen_best.mask.copy(), dict(gen_best.params),
                                  gen_best.fitness, gen_best.raw_score)
            self.cfg.history.append({"generation": gen, "best_raw": float(gen_best.raw_score),
                                     "n_features": int(gen_best.mask.sum())})
            if self.cfg.verbose:
                print(f"    generation {gen:2d}: best={gen_best.raw_score:.4f} "
                     f"(features: {int(gen_best.mask.sum())})")
        return best


GA_PARAM_SPACE_REG = {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 4, 6],
                      "min_samples_leaf": [1, 2, 3, 5], "max_features": [0.4, 0.6, 0.8, 1.0]}
GA_PARAM_SPACE_CLF = {"n_estimators": [150, 300], "max_depth": [3, 4, 6],
                      "min_samples_leaf": [1, 2, 3], "max_features": [0.4, 0.6, 0.8]}


def make_ga_regressor(params: dict) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=int(params["n_estimators"]), max_depth=int(params["max_depth"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_features=float(params["max_features"]), random_state=RANDOM_STATE, n_jobs=1)


def make_ga_classifier(params: dict) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=int(params["n_estimators"]), max_depth=int(params["max_depth"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_features=float(params["max_features"]), class_weight="balanced_subsample",
        random_state=RANDOM_STATE, n_jobs=1)


def run_ga(X, y, groups, task="reg", generations=15):
    cols = list(X.columns)
    Xv = X.to_numpy()
    key = "R2" if task == "reg" else "macro_F1"
    metric_fn = regression_metrics if task == "reg" else classification_metrics
    make = make_ga_regressor if task == "reg" else make_ga_classifier
    space = GA_PARAM_SPACE_REG if task == "reg" else GA_PARAM_SPACE_CLF
    # running LOSO inside the GA loop (40 folds x 30 individuals x 15
    # generations) would be prohibitively slow; a 5-fold grouped CV is used
    # instead, and an honest LOSO evaluation of the final selected
    # configuration is computed separately below
    inner = GroupKFold(n_splits=5)

    def score_fn(mask, params):
        try:
            pred = cross_val_predict(make(params), Xv[:, mask], y,
                                     groups=groups, cv=inner, n_jobs=-1)
            return metric_fn(y, pred)[key]
        except Exception:
            return -1.0

    ga = EnhancedGA(len(cols), space, score_fn, EGAConfig(generations=generations, random_state=42))
    best = ga.run()
    selected = [c for c, m in zip(cols, best.mask) if m]
    print(f"    selected {len(selected)} features out of {len(cols)}: {selected}")
    print(f"    hyperparameters: {dict(best.params)}")
    return selected, {k: (int(v) if k != "max_features" else float(v))
                      for k, v in best.params.items()}, ga.cfg.history


In [ ]:
sel_b, params_b, hist_b = run_ga(Xb, yb, groups, "reg")
res_ga_b = evaluate(make_ga_regressor(params_b), Xb[sel_b], yb, groups, blocks, "reg")
res_ga_b.pop("_pred_site")
print(f"\nGA model, by site:   {fmt_metrics(res_ga_b['by_site'])}")
print(f"GA model, by region: {fmt_metrics(res_ga_b['by_region'])}")
res_b["ega"] = {"features": sel_b, "params": params_b, "metrics": res_ga_b, "history": hist_b}


<div class="finding">
This result is notable: <b>six features instead of twenty-three</b>, with R²
by site increasing from 0.73 to 0.78, and by region from 0.59 to 0.71. The
improvement by region is larger than by site, indicating that the selection
did not simply overfit to familiar data — it removed something that was
actively hindering generalization.

The selected set is <code>bd, elevation_m, land_use_grass cover, ph_sn,
precip_mm, sm_grav</code>. Compared against the referenced NPK study, which
predicts nitrogen from temperature, moisture, pH, and precipitation, the GA
arrived at a nearly identical set independently, substituting elevation for
temperature — a reasonable substitution for a transect where elevation and
temperature are tightly coupled, and elevation additionally captures terrain
effects.
</div>


## Organic carbon

The same feature set used for nitrogen applies here, so the baseline step is
not repeated — its behavior on this feature set is already established. The
comparison proceeds directly to the full model set and the GA.


In [ ]:
Xc = encode(df, num_nutrient, CATEGORICAL_FEATURES)
yc = df["toc"].to_numpy()
best_c, res_c = compare_zoo("ORGANIC CARBON, full model comparison", Xc, yc, groups, blocks,
                            regressors(), "reg")

sel_c, params_c, hist_c = run_ga(Xc, yc, groups, "reg")
res_ga_c = evaluate(make_ga_regressor(params_c), Xc[sel_c], yc, groups, blocks, "reg")
res_ga_c.pop("_pred_site")
print(f"\nGA model, by site:   {fmt_metrics(res_ga_c['by_site'])}")
print(f"GA model, by region: {fmt_metrics(res_ga_c['by_region'])}")
res_c["ega"] = {"features": sel_c, "params": params_c, "metrics": res_ga_c, "history": hist_c}


<div class="note">
Here the GA provides almost no improvement over plain XGBoost (0.68 vs. 0.69
by site), so the full feature set is used in production rather than the
reduced one. The selected subset is nonetheless informative:
<code>bd, has_solonetz, land_use_crop/shrub cover, sm_grav,
soil_group_kastanozem/saline</code>. Notably, no climate feature is
included. For nitrogen the GA consistently selected precipitation; for
carbon it selected none. This suggests that soil organic carbon stock
reflects accumulated, longer-term history — soil type and land-use legacy —
rather than current weather, whereas nitrogen responds more directly to the
present precipitation regime. This interpretation has not been tested
rigorously, but is plausible given the pattern.
</div>


## Primary model: soil condition classifier

This is the model the target application actually requires. As with the
regression targets, the process begins with a simple baseline — logistic
regression on the full feature set.


In [ ]:
Xd = encode(df, num_nutrient, CATEGORICAL_FEATURES)
yd = df["soil_state_idx"].to_numpy()

baseline_clf = Pipeline([("scaler", StandardScaler()),
                         ("model", LogisticRegression(C=1.0, max_iter=2000, random_state=RANDOM_STATE))])
res_baseline_clf = evaluate(baseline_clf, Xd, yd, groups, blocks, "clf")
print(f"logistic regression: by site   {fmt_metrics(res_baseline_clf['by_site'])}")
print(f"                     by region {fmt_metrics(res_baseline_clf['by_region'])}")


A macro-F1 of approximately 0.59 is unremarkable — not poor, but not
sufficient to justify the full pipeline on its own. A broader classifier
comparison follows.


In [ ]:
def classifiers() -> dict:
    return {
        "LogisticRegression": _scaled(LogisticRegression(
            C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=4, class_weight="balanced",
            random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=500, max_depth=6, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=1),
        "KNN": _scaled(KNeighborsClassifier(n_neighbors=5, weights="distance")),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
            min_child_weight=2, random_state=RANDOM_STATE, n_jobs=1, verbosity=0),
    }


best_d, res_d = compare_zoo("SOIL CONDITION CLASS, full model comparison", Xd, yd, groups, blocks,
                            classifiers(), "clf")


RandomForest raises macro-F1 to 0.71 — a meaningful improvement. This is the
point at which the caveat raised during label construction needs to be
addressed directly: since pH and EC enter the scoring formula itself, how
much of this performance reflects the model recovering the formula rather
than learning new structure?


In [ ]:
Xd_ctrl = encode(df, ["ph_sn", "ec_log"], None)
res_ctrl = evaluate(classifiers()["RandomForest"], Xd_ctrl, yd, groups, blocks, "clf")
res_ctrl.pop("_pred_site")
print(f"pH and EC only: {fmt_metrics(res_ctrl['by_site'])}")
res_d["control_ph_ec_only"] = res_ctrl


<div class="warn">
A model trained on pH and EC alone already achieves macro-F1=0.52 — this is
the portion of performance attributable to formula-level leakage. The full
model, using all features, achieves 0.71. The genuine, non-trivial
contribution of the remaining twenty features is therefore the difference,
approximately <b>+0.19</b>. The model does not recover soil condition from
nothing: a substantial share of the signal originates in the label's own
construction, and this should be stated explicitly rather than obscured
behind a single aggregate performance figure.
</div>

The GA is applied next, with feature-set size penalized as before: 12
generations rather than 15, since with only four classes convergence occurs
sooner and additional generations are not justified.


In [ ]:
sel_d, params_d, hist_d = run_ga(Xd, yd, groups, "clf", generations=12)
res_ga_d = evaluate(make_ga_classifier(params_d), Xd[sel_d], yd, groups, blocks, "clf")
pred_ga_d = res_ga_d.pop("_pred_site")
print(f"\nGA model, by site:   {fmt_metrics(res_ga_d['by_site'])}")
print(f"GA model, by region: {fmt_metrics(res_ga_d['by_region'])}")
res_d["ega"] = {"features": sel_d, "params": params_d, "metrics": res_ga_d, "history": hist_d}

cm = confusion_matrix(yd, pred_ga_d)
res_d["confusion_matrix"] = cm.tolist()


<div class="finding">
The GA reaches macro-F1=0.75 using ten features, outperforming the entire
model comparison, including the full 23-feature set. After accounting for
the pH/EC leakage, the genuine model contribution is therefore somewhat
larger than the 0.71 vs. 0.52 comparison above suggested: the GA identified
a more effective feature combination than a random forest trained on the
full feature set.
</div>


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(cm, cmap=LinearSegmentedColormap.from_list("acc", ["white", ACCENT]))
ax.set_xticks(range(4)); ax.set_xticklabels(SOIL_STATE_CLASSES, rotation=20, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(SOIL_STATE_CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix — GA · RandomForest (LOSO)", loc="left")
for i in range(4):
    for j in range(4):
        v = cm[i, j]
        ax.text(j, i, str(v), ha="center", va="center",
               color="white" if v > cm.max() * 0.5 else "#191B14",
               fontweight="bold" if i == j else "normal")
ax.grid(False)
plt.tight_layout()
plt.show()


<div class="finding">
The upper-left corner is the result that matters most for establishing trust
in this model: <b>no critical-condition site is misclassified as moderate or
healthy</b>. Where the model does err on critical sites, it errs toward
"poor" — the conservative direction, not the reverse.
</div>

The dominant confusion is between `poor` and `moderate`. This likely
reflects a limitation of the label rather than the model: a cutoff of 10
points out of 18 is a somewhat arbitrary boundary, and drawing a sharp line
near it is difficult even manually.


## Soil pH — an honest negative result

[A comparable project predicting soil pH](https://github.com/riponalmamun/Prediction-of-Soil-pH)
reports a best result (Random Forest) of R²=0.62 and RMSE=0.52 on its own
data, with iron, calcium carbonate, and manganese as the leading predictors.
Iron and manganese are not available in this dataset, but carbonate content
is available directly — TIC (inorganic carbon), from the first sheet. The
same algorithm set (SVR, a neural network, and stacking) is used here for
comparison.


In [ ]:
def ph_regressors() -> dict:
    base = regressors()
    base["SVR"] = _scaled(SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale"))
    base["ANN"] = _scaled(MLPRegressor(
        hidden_layer_sizes=(32,), alpha=1.0, max_iter=3000,
        early_stopping=True, n_iter_no_change=30, random_state=RANDOM_STATE))
    # cv=3 and fewer trees than usual: stacking refits base models inside
    # every outer LOSO fold, and at full scale this takes hours rather than
    # minutes
    base["Stacking"] = StackingRegressor(
        estimators=[
            ("rf", RandomForestRegressor(n_estimators=150, max_depth=6,
                min_samples_leaf=3, max_features="sqrt",
                random_state=RANDOM_STATE, n_jobs=1)),
            ("svr", _scaled(SVR(kernel="rbf", C=10.0, epsilon=0.1))),
            ("gbr", GradientBoostingRegressor(n_estimators=200, max_depth=2,
                learning_rate=0.05, random_state=RANDOM_STATE)),
        ], final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=1)
    return base


num_ph_full = (["ec_log", "sm_grav", "bd", "tic", "toc", "tn", "carbonate_frac"]
              + CLIMATE_FEATURES + TERRAIN_FEATURES + SEASON_FEATURES + BINARY_FEATURES)
Xe = encode(df, num_ph_full, CATEGORICAL_FEATURES)
ye = df["ph_sn"].to_numpy()
best_e, res_e = compare_zoo("SOIL pH, with laboratory chemistry", Xe, ye, groups, blocks,
                            ph_regressors(), "reg")
print("\nreference from external repository: R2=0.62, RMSE=0.52")


<div class="fail">
The best result here is XGBoost, at R²=0.40 against the reference's 0.62 —
short of the target. The hypothesis that carbonate content could substitute
for CaCO₃ is only partially supported: a signal is present, but one feature
out of the three strongest predictors in the reference study is evidently
not sufficient to close the gap. This shortfall is reported as is, rather
than reframed as a success.
</div>

The following check is the reason a neural network was tried on this dataset
at all — to see whether an MLP would surface any signal that tree-based
models do not.


In [ ]:
print("ANN R2 by site:  ", round(res_e["algorithms"]["ANN"]["by_site"]["R2"], 3))
print("ANN R2 by region:", round(res_e["algorithms"]["ANN"]["by_region"]["R2"], 3))


<div class="fail">
R²=-1.88 by site and -6.20 by region. The convergence warnings emitted
during the model comparison above are the explanation: scikit-learn
reported, on every one of the 40 folds, that the optimizer failed to
converge within 3000 iterations. This is not a hyperparameter issue — it is
the expected outcome of attempting to train a neural network on 80 rows:
there is simply not enough data to reach a stable minimum, and the optimizer
settles in a different location on each fold. This result is reported as
is; a transparent negative result is more useful here than a silently
dropped row.
</div>

A version that does not rely on laboratory chemistry is also needed in
practice, since a pH electrode can fail in the field, and some estimate
should remain available using only what a sensor and site coordinates can
provide.


In [ ]:
num_ph_sensor = (["ec_log", "sm_grav", "bd"] + CLIMATE_FEATURES + TERRAIN_FEATURES
                + SEASON_FEATURES + BINARY_FEATURES)
Xe_s = encode(df, num_ph_sensor, CATEGORICAL_FEATURES)
best_es, res_es = compare_zoo("pH WITHOUT LABORATORY CHEMISTRY (production variant)",
                              Xe_s, ye, groups, blocks, ph_regressors(), "reg")
res_e["sensor_only"] = {"algorithms": res_es["algorithms"], "best": res_es["best"]}


As expected, performance drops further (R²=0.34) — removing the laboratory
chemistry features removes part of the signal along with it. This variant is
nonetheless the one used in production, since it is the only one that does
not require a laboratory.


## Consolidated results

All five models are compared side by side, with particular attention to how
far the by-site and by-region scores diverge for each.


In [ ]:
summary_rows = [
    ("Total nitrogen", "tn",            "GA · RandomForest", len(sel_b), res_ga_b["by_site"], res_ga_b["by_region"], "R2"),
    ("Organic carbon", "toc",           best_c,                    Xc.shape[1], res_c["algorithms"][best_c]["by_site"], res_c["algorithms"][best_c]["by_region"], "R2"),
    ("Moisture",       "sm_grav",       best_a,                    Xa.shape[1], res_a["algorithms"][best_a]["by_site"], res_a["algorithms"][best_a]["by_region"], "R2"),
    ("Soil condition", "4 classes",     "GA · RandomForest", len(sel_d), res_ga_d["by_site"], res_ga_d["by_region"], "macro_F1"),
    ("pH",             "ph_sn (sensor)",best_es,                   Xe_s.shape[1], res_es["algorithms"][best_es]["by_site"], res_es["algorithms"][best_es]["by_region"], "R2"),
]

summary = pd.DataFrame([{
    "Model": name, "Target": target, "Algorithm": algo, "Features": nf,
    "LOSO": f"{key} {site[key]:.3f}", "By region": f"{key} {region[key]:.3f}",
} for name, target, algo, nf, site, region, key in summary_rows])
summary


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
names = [r[0] for r in summary_rows]
site_v = [r[4][r[6]] for r in summary_rows]
reg_v = [r[5][r[6]] for r in summary_rows]
y_pos = np.arange(len(names))[::-1]

for yp, s, rg in zip(y_pos, site_v, reg_v):
    ax.plot([rg, s], [yp, yp], color=ACCENT_DIM, lw=2, zorder=1)
ax.scatter(reg_v, y_pos, s=70, facecolor="white", edgecolor=ACCENT_DIM, lw=2,
          zorder=2, label="by region")
ax.scatter(site_v, y_pos, s=70, color=ACCENT, edgecolor="white", lw=1.5,
          zorder=3, label="by site (LOSO)")
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel("R² / macro-F1 on held-out folds")
ax.set_title("Where the model generalizes honestly, versus where it relies on spatial proximity", loc="left")
ax.set_xlim(0, 0.85)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()


<div class="note">
For nitrogen the segment is barely visible — the by-site and by-region
scores are nearly identical. This is the model with the highest degree of
trust: it appears to have learned an underlying physical relationship rather
than exploiting geographic proximity. For the other targets the segment is
longer, and its length is read here as an honest measure of how much
"interpolation between neighbors" is hidden behind an otherwise favorable
LOSO score.
</div>


In [ ]:
all_feats = sorted(set(sel_b) | set(sel_c) | set(sel_d))
sel_matrix = pd.DataFrame({
    "Total nitrogen": [f in sel_b for f in all_feats],
    "Organic carbon": [f in sel_c for f in all_feats],
    "Soil condition": [f in sel_d for f in all_feats],
}, index=all_feats).astype(int)

fig, ax = plt.subplots(figsize=(6, max(3.5, 0.32 * len(all_feats))))
ax.imshow(sel_matrix.T, cmap=LinearSegmentedColormap.from_list("dot", ["white", ACCENT]),
         aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(all_feats)))
ax.set_xticklabels(all_feats, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(3))
ax.set_yticklabels(sel_matrix.columns)
ax.set_title("Features selected by the genetic algorithm, by target", loc="left")
ax.grid(False)
plt.tight_layout()
plt.show()


<div class="finding">
The carbon row in this figure is the result that prompted a second look:
<b>no climate feature is selected at all</b>. This was initially suspected
to be a code error, but on inspection it is not — the selection is correct.
Soil organic carbon stock reflects accumulated history rather than a
response to present-day weather, unlike nitrogen, which clearly responds to
the current precipitation regime. Each fact was known individually
beforehand; seeing the GA separate them into distinct feature sets without
any explicit guidance is a useful confirmation.
</div>


## Saving trained models

Models are written to `/kaggle/working/ml_models/`, using the same format
expected by the project backend: a `<name>_model.pkl` / `<name>_meta.pkl`
pair, with the meta file storing the feature list and evaluation metrics.


In [ ]:
import joblib

MODELS_DIR = Path("/kaggle/working/ml_models") if Path("/kaggle/working").exists() else Path("ml_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)


def pick(res, X, y, task):
    key = "R2" if task == "reg" else "macro_F1"
    zoo_best = res["algorithms"][res["best"]]["by_site"][key]
    if "ega" in res and res["ega"]["metrics"]["by_site"][key] >= zoo_best:
        feats = res["ega"]["features"]
        mk = make_ga_regressor if task == "reg" else make_ga_classifier
        model = mk(res["ega"]["params"])
        return model.fit(X[feats], y), feats, "GA-RandomForest", res["ega"]["metrics"]["by_site"]
    name = res["best"]
    model = (regressors() if task == "reg" else classifiers())[name]
    return model.fit(X, y), list(X.columns), name, res["algorithms"][name]["by_site"]


for tag, res, X, y, task, fname in [
    ("moisture", res_a, Xa, ya, "reg", "soil_moisture_model.pkl"),
    ("nitrogen", res_b, Xb, yb, "reg", "soil_nitrogen_model.pkl"),
    ("carbon",   res_c, Xc, yc, "reg", "soil_carbon_model.pkl"),
    ("state",    res_d, Xd, yd, "clf", "soil_state_model.pkl"),
    ("ph",       res_es, Xe_s, ye, "reg", "soil_ph_model.pkl"),
]:
    model, feats, algo, metrics = pick(res, X, y, task)
    joblib.dump(model, MODELS_DIR / fname)
    meta = {
        "features": feats, "algorithm": algo, "task": task, "metrics_loso": metrics,
        "target": {"moisture": "sm_grav", "nitrogen": "tn", "carbon": "toc",
                   "state": "soil_state_idx", "ph": "ph_sn"}[tag],
        "n_train": int(len(y)), "n_sites": int(df["sample_id"].nunique()),
        "dataset": "Supplement 2.xlsx (Kazakhstan transect, 2015)",
    }
    if task == "clf":
        meta["classes"] = SOIL_STATE_CLASSES
    joblib.dump(meta, MODELS_DIR / fname.replace("_model.pkl", "_meta.pkl"))
    print(f"  {fname:<28} {algo:<20} features={len(feats):<3} {fmt_metrics(metrics)}")


## Limitations

- The model performs well **within** this transect. Outside its geographic
  range, classifier performance drops by nearly half (macro-F1 0.75 -> 0.42)
  — this is not a minor caveat but a direct reason not to trust predictions
  outside Kazakhstan without retraining on local data.
- Nitrogen is measured without corresponding phosphorus or potassium data,
  so this does not constitute a full NPK assessment.
- The pH model is not a substitute for a pH electrode; at best it is a
  coarse indicator that a sensor reading may be off.
- The 80 rows correspond to 40 sites, not 80 independent observations. Any
  confidence estimate derived from this dataset should be interpreted with
  that qualification in mind.

A comparable transect in another region would be a valuable test of whether
the nitrogen-climate and carbon-land-use-history relationships observed here
generalize, or are specific to the Kazakh steppe. Contributions and
replications of this analysis are welcome.
